# Validação `portopt` × Datasets Chagas

Este notebook reproduz os exercícios principais do curso usando os datasets oficiais (`Ex1.xlsx`, `MDR_Example.xlsx`, `MCVaR_Example.xls`) e mostra que o `portopt` produz resultados consistentes com os apresentados pelo Prof. Chagas.

Os 3 datasets vêm com o pacote (`portopt.datasets`) e podem ser usados diretamente.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import portopt as po
from portopt import datasets
from portopt.costs import FlatCost

%matplotlib inline
pd.options.display.float_format = '{:.4f}'.format

# Lista de datasets disponíveis
for name in datasets.list_datasets():
    info = datasets.info(name)
    print(f"\n[{name}] {info['description']}")
    print(f"  Período: {info['period']}")
    print(f"  Exercício: {info['exercise']}")

## 1. Ex1.xlsx — Ações brasileiras (nb1: MV vs EW backtest)

Reproduz o experimento didático central do nb1: comparar BH/EW/MV sob custos de transação.

In [ ]:
br_prices = datasets.subset('ex1', 'br_stocks').loc['2014':]
print(f'Universo: {br_prices.shape[1]} ações BR, período {br_prices.index[0].date()} → {br_prices.index[-1].date()}')

comparison = po.compare(
    models=['ew', 'markowitz', 'mad', 'erc', 'hrp'],
    prices=br_prices,
    constraints=po.ConstraintSet(bounds=(0.0, 1.0)),
    with_backtest=True,
    backtest_config=po.BacktestConfig(
        training_window=5 * 252,
        rebalance='monthly',
        transaction_costs=FlatCost(rate=0.0015),  # 15 bps Chagas standard
        progress=False,
    ),
)
comparison.metrics_table().T[['annualized_return', 'annualized_vol', 'sharpe',
                              'max_drawdown', 'ulcer_index']]

In [ ]:
# Plot cumulative wealth
fig, ax = plt.subplots(figsize=(11, 4))
comparison.cumulative_wealth().plot(ax=ax, linewidth=1.3)
ax.set_title('Backtest BR equities (Ex1.xlsx, 2014-2023, 5y window, 15bps)')
ax.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout()

# A lição do Chagas (nb1 §2.7): MV não vence EW out-of-sample neste universo
ew_final = comparison.cumulative_wealth()['ew'].iloc[-1]
mv_final = comparison.cumulative_wealth()['markowitz'].iloc[-1]
print(f'\nEW final wealth: {ew_final:.2f}, MV final wealth: {mv_final:.2f}')
print(f'EW > MV em retorno absoluto: {ew_final > mv_final}  ← lição central do Chagas')

## 2. MDR_Example.xlsx — Downside Risk em commodities (nb2)

MV, MAD e DR produzem alocações similares quando os retornos são razoavelmente simétricos.

In [ ]:
# Chagas usa 5d log-returns no nb2
mdr_prices = datasets.load('mdr')
mdr_weekly = mdr_prices.iloc[::5]
mdr_logrets = po.returns.to_log_returns(mdr_weekly)

result = po.compare(
    models=['ew', 'markowitz', 'mad', 'downside_risk', 'erc', 'hrp'],
    prices=mdr_weekly,
    log_returns=mdr_logrets,
    constraints=po.ConstraintSet(bounds=(0.0, 1.0)),
)
result.weights_table().style.format('{:.3f}').background_gradient(cmap='YlOrRd', vmin=0, vmax=0.3)

## 3. MCVaR_Example.xls — CVaR em commodities (nb2)

Chagas usa 10.000 cenários multinormais simulados a partir de μ, Σ do sample. Comparamos CVaR-MVN ↔ CVaR-histórico ↔ MV.

**Lição do PDF §3.5**: as MVN simulations subestimam a cauda, então os pesos CVaR se parecem demais com os do MV. Reproduzimos exatamente.

In [ ]:
mcvar_prices = datasets.load('mcvar').iloc[::5]
mcvar_logrets = po.returns.to_log_returns(mcvar_prices)

results = {}
results['MV (reference)'] = po.models.Markowitz().fit(mcvar_logrets, po.ConstraintSet())
results['CVaR (10k MVN sim)'] = po.models.CVaR(alpha=0.05, n_scenarios=10_000,
                                                random_state=42).fit(mcvar_logrets, po.ConstraintSet())
results['CVaR (historical)'] = po.models.CVaR(alpha=0.05, n_scenarios=0).fit(
    mcvar_logrets, po.ConstraintSet())

df = pd.DataFrame({k: r.weights for k, r in results.items()})
df = df[df.max(axis=1) > 0.01].sort_values(by='MV (reference)', ascending=False)
ax = df.plot.barh(figsize=(10, 5), width=0.8)
ax.set_title('CVaR (Chagas §3.5): MVN simulations underestimate tail → weights ≈ MV')
ax.invert_yaxis()
ax.grid(True, axis='x', linestyle='--', linewidth=0.5)
plt.tight_layout()

## 4. Risk Budgeting com grupos de commodities (nb3)

Reproduz a partição Metais / Energia / Agricultura / Livestock com risk budgets explícitos.

In [ ]:
# Constraints com grupos de Chagas
constraints = po.ConstraintSet(
    bounds=(0.0, 1.0),
    asset_groups={
        'Metals': datasets.ASSET_GROUPS['mdr']['metals'],
        'Energy': datasets.ASSET_GROUPS['mdr']['energy'],
        'Agri': datasets.ASSET_GROUPS['mdr']['agri'],
        'Livestock': datasets.ASSET_GROUPS['mdr']['livestock'],
    },
    group_risk_budgets={'Metals': 0.20, 'Energy': 0.30, 'Agri': 0.30, 'Livestock': 0.20},
)

# Comparação Approach 1 vs Approach 2
rb1 = po.models.RiskBudget(approach='1').fit(mdr_logrets, constraints)
rb2 = po.models.RiskBudget(approach='2').fit(mdr_logrets, constraints)

print('=== RB targets vs realized ===')
for label, r in [('Approach 1 (hard)', rb1), ('Approach 2 (soft)', rb2)]:
    print(f'\n{label}: converged={r.converged}')
    tgt = r.diagnostics['group_budgets_target']
    real = r.diagnostics['group_budgets_realized']
    for g in tgt:
        print(f'  {g:10s}  target={tgt[g]:.2f}  realized={real[g]:.4f}')

## 5. HRP com dendrograma (nb3)

O Lopez de Prado clássico: clustering + quasi-diagonalização + recursive bisection.

In [ ]:
hrp = po.models.HierarchicalRiskParity(return_dendrogram=True).fit(
    mdr_logrets, po.ConstraintSet()
)
linkages = np.array(hrp.diagnostics['linkages'])
qd_names = hrp.diagnostics['qd_order_names']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
po.viz.plot_hrp_dendrogram(linkages, list(mdr_logrets.columns), ax=axes[0])
hrp.weights.sort_values().plot.barh(ax=axes[1], color='steelblue')
axes[1].set_title('HRP weights')
axes[1].grid(True, axis='x', linestyle='--', linewidth=0.5)
plt.tight_layout()

print(f'\nQuasi-diagonalization order: {qd_names[:5]} ... {qd_names[-3:]}')